# Fine-tuning sobre Nutrition5k autoetiquetado — cabeza CLIP (y B1 opcional)

Entrena y evalúa los clasificadores de región con las etiquetas que exportó el notebook de autoetiquetado. **No importa nada de ese notebook**: solo lee carpetas del disco.

**Entradas**
- `dataset_A_minimo/` y `dataset_B_estricto/` (cada una con `labels/`, `classes.json`, `manifest.csv`, `class_stats.csv`).
- `clean_refs/`: referencias del set limpio **con los ids ya remapeados** (se generan una vez, ver "Antes de empezar").

**Orden de trabajo (en cascada)**
1. Cabeza lineal sobre embeddings CLIP de región, inicializada con el texto de CLIP (rápido; CLIP queda congelado).
2. Evaluación en el set limpio, con regiones de referencia y con regiones predichas por B1.
3. **Solo si la evaluación lo pide**, fine-tuning de B1 (Parte 7, apagada por defecto).

| Parte | Contenido |
|---|---|
| 1–2 | Configuración, datos y verificaciones |
| 3 | CLIP, regiones y embeddings (con caché) |
| 4 | Cabeza: definición, barrido de regularización y entrenamiento (3 semillas) |
| 5 | Evaluación en el set limpio (A frente a B, bootstrap por plato) |
| 6 | Guardar cabezas |
| 7 | (Opcional) Fine-tuning de B1 |

**Reglas de lectura**
- Compara A y B **solo en el set limpio**. La validación (10 % de platos) sale de las mismas etiquetas ruidosas de cada versión y no es comparable entre A y B.
- Con 40 a 60 platos de referencia, diferencias de uno o dos puntos son ruido. Usa la tabla con ± entre semillas y el intervalo del bootstrap.

## Antes de empezar: exportar las referencias del set limpio

Hazlo **una vez, en el notebook de autoetiquetado**, con `lut` y `class_names_final` ya definidos (los de la celda de remapeo). Las máscaras del set limpio usan los ids originales; esta celda las pasa a los ids finales para que coincidan con los datasets exportados.

```python
def export_clean_refs(out_dir="clean_refs"):
    out = Path(out_dir); (out / "labels").mkdir(parents=True, exist_ok=True)
    (out / "classes.json").write_text(json.dumps(
        {"classes": class_names_final, "ignore_index": IGNORE}, indent=1), encoding="utf-8")
    rows = []
    for group, cdir in [("clean", "clean_set"), ("val", "clean_set_val")]:
        if not (Path(cdir) / "regions_gt.csv").exists():
            continue
        for d, sem in load_clean_gt(cdir).items():
            Image.fromarray(lut[sem]).save(out / "labels" / f"{d}.png")
            rows.append(dict(dish=d, label=str(Path("labels") / f"{d}.png"), group=group))
    pd.DataFrame(rows).to_csv(out / "manifest.csv", index=False)
    print(len(rows), "platos de referencia ->", out)

export_clean_refs()
```

Luego copia la carpeta `clean_refs/` junto a este notebook (o ajusta `REFS_DIR`).

## 1. Configuración

In [1]:
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------- RUTAS ----------------
ROOT     = Path(r"C:\nutrition5k")                 # imágenes: ROOT/realsense_overhead/<plato>/rgb.png
SEG_PATH = r"C:\foodseg_segformer\mit-b1"          # B1 entrenado en FoodSeg103
DATASETS = {"A": Path("dataset_A_minimo"), "B": Path("dataset_B_estricto")}
REFS_DIR = Path("clean_refs")                      # referencias del set limpio (ver "Antes de empezar")
CACHE    = Path("cache_ft"); CACHE.mkdir(exist_ok=True)

# ---------------- PARÁMETROS ----------------
device        = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_NAME     = "openai/clip-vit-large-patch14"
CLIP_CHUNK    = 16                 # recortes por lote en CLIP (baja si te quedas sin VRAM)
INPUT_SIZE    = (512, 512)         # tamaño con el que se entrenó B1
MIN_REGION_PX = 1500               # mismo mínimo de área que usó el etiquetador
IGNORE        = 255
SEEDS         = [0, 1, 2]
TAUS          = (0.0, 0.3, 0.5)    # umbral de la cabeza: por debajo, la región queda "sin etiqueta"
TAU_MAIN      = 0.3                # el usado para el bootstrap y el informe por clase
MAX_DISHES    = None               # p. ej. 60 para una prueba rápida (usa su propia caché)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def _p(x):                         # rutas del manifest escritas en Windows (con "\\") también funcionan en Linux
    return Path(str(x).replace("\\", "/"))

def img_path(dish):
    return ROOT / "realsense_overhead" / dish / "rgb.png"

def load_rgb(dish):
    return np.array(Image.open(img_path(dish)).convert("RGB"))

print("device:", device)

device: cuda


## 2. Datos y verificaciones

Comprueba que A, B y las referencias comparten vocabulario y que **ningún plato de referencia está en train o val** (la fuga que corregimos en la exportación).

In [2]:
def load_dataset(path):
    path = Path(path)
    man = pd.read_csv(path / "manifest.csv", keep_default_na=False, dtype={"dish": str, "label": str, "split": str})
    classes = json.loads((path / "classes.json").read_text(encoding="utf-8"))["classes"]
    stats = pd.read_csv(path / "class_stats.csv")
    return man, classes, stats

DS = {}
for tag, p in DATASETS.items():
    man, classes, stats = load_dataset(p)
    DS[tag] = dict(dir=p, man=man, classes=classes, stats=stats)
    print(f"{tag}: {p} | splits: {man.split.value_counts().to_dict()} | clases: {len(classes) - 1}")
    print("   motivos de exclusión:", man[man.reason != ""].reason.value_counts().to_dict())

CLASSES = DS["A"]["classes"]                       # ["background", clase1, ..., claseK]
assert all(DS[t]["classes"] == CLASSES for t in DS), "A y B tienen vocabularios distintos: re-exporta con el mismo lut"
K = len(CLASSES) - 1

refs = pd.read_csv(REFS_DIR / "manifest.csv", keep_default_na=False, dtype=str)
assert json.loads((REFS_DIR / "classes.json").read_text(encoding="utf-8"))["classes"] == CLASSES, \
    "clean_refs usa otro vocabulario: vuelve a exportarlo con el lut actual"
for t in DS:
    used = set(DS[t]["man"].query("split in ['train', 'val']").dish)
    clash = used & set(refs.dish)
    assert not clash, f"{len(clash)} platos de referencia están en train/val de {t} (ej. {sorted(clash)[:3]}): re-exporta con exclude"
print("referencias:", refs.group.value_counts().to_dict(), "| sin solapamiento con train/val")

for t in DS:
    s = DS[t]["stats"]
    print(f"{t}: {int((s.platos >= 30).sum())} clases con ≥30 platos | {int(((s.platos >= 10) & (s.platos < 30)).sum())} con 10–29 | {int((s.platos < 10).sum())} con <10")

A: dataset_A_minimo | splits: {'train': 1120, '': 125, 'val': 122, 'test': 40} | clases: 67
   motivos de exclusión: {'low_coverage': 125, 'test': 40}
B: dataset_B_estricto | splits: {'train': 991, '': 271, 'val': 105, 'test': 40} | clases: 67
   motivos de exclusión: {'dropped_quality': 153, 'low_coverage': 118, 'test': 40}
referencias: {'clean': 40} | sin solapamiento con train/val
A: 35 clases con ≥30 platos | 30 con 10–29 | 2 con <10
B: 32 clases con ≥30 platos | 28 con 10–29 | 7 con <10


### Utilidades sin GPU

Regiones (componentes conectados por clase) y métricas. Son las mismas definiciones del `evaluate` del notebook de autoetiquetado: *cobertura*, *precisión*, *recall correcto* y *mIoU*, solo sobre píxeles con referencia (`gt != 255`).

In [3]:
def components(class_map, min_px=MIN_REGION_PX):
    """Componentes conectados de cada clase (ignora 0 = fondo y 255). -> (region_map uint16, clase de origen de cada región)."""
    rm = np.zeros(class_map.shape, np.uint16); src = []
    for c in np.unique(class_map):
        if c == 0 or c == IGNORE:
            continue
        comp, n = ndimage.label(class_map == c)
        if n == 0:
            continue
        areas = np.bincount(comp.ravel())[1:]
        for k in np.nonzero(areas >= min_px)[0] + 1:
            src.append(int(c)); rm[comp == k] = len(src)
    return rm, src

def conf_from_map(pred, gt):
    """Matriz de confusión (K+2)x(K+2): filas = referencia 0..K, columnas = predicción 0..K y K+1 = comida sin etiqueta."""
    n = K + 2
    valid = gt != IGNORE
    idx = gt[valid].astype(np.int64) * n + pred[valid].astype(np.int64)
    return np.bincount(idx, minlength=n * n).reshape(n, n)

def conf_for_dish(pr, gt, probs, tau):
    """pr: regiones predichas del plato (dict con 'rm' y 'n'); probs: (n, K) de la cabeza."""
    U = K + 1
    lut = np.full(pr["n"] + 1, U, np.int64); lut[0] = 0
    if pr["n"]:
        lut[1:] = np.where(probs.max(1) >= tau, probs.argmax(1) + 1, U)
    return conf_from_map(lut[pr["rm"].astype(np.int64)], gt)

def metrics_from_conf(conf):
    n = conf.shape[0]; Kc = n - 2
    gt_food = max(int(conf[1:Kc + 1].sum()), 1)
    tp = np.diag(conf)[1:Kc + 1]
    pred_as_class = max(int(conf[:, 1:Kc + 1].sum()), 1)
    iou = iou_from_conf(conf)
    return dict(cobertura=conf[1:Kc + 1, 1:Kc + 1].sum() / gt_food, precision=tp.sum() / pred_as_class,
                recall_correcto=tp.sum() / gt_food, mIoU=float(np.nanmean(iou)) if np.isfinite(iou).any() else np.nan)

def iou_from_conf(conf):
    Kc = conf.shape[0] - 2
    tp = np.diag(conf)[1:Kc + 1]
    gt_c, pr_c = conf[1:Kc + 1].sum(1), conf[:, 1:Kc + 1].sum(0)
    return np.where(gt_c > 0, tp / np.maximum(gt_c + pr_c - tp, 1), np.nan)

def metrics_p1(probs, y, area):
    """Clasificación con regiones de referencia: y = clase real (1..K) de cada región."""
    ok = (probs.argmax(1) + 1) == y
    rec = [ok[y == c].mean() for c in np.unique(y)]
    return dict(acc_px=float((area * ok).sum() / area.sum()), acc_region=float(ok.mean()),
                recall_medio=float(np.mean(rec)), n_regiones=int(len(y)))

def boot_diff(confs_a, confs_b, n_boot=1000, seed=0):
    """Bootstrap por plato de mIoU(B) - mIoU(A). confs_*: lista (una por semilla) de arrays (n_platos, K+2, K+2)."""
    rng = np.random.default_rng(seed); N = confs_a[0].shape[0]; d = []
    for _ in range(n_boot):
        idx = rng.integers(0, N, N)
        ma = np.mean([metrics_from_conf(c[idx].sum(0))["mIoU"] for c in confs_a])
        mb = np.mean([metrics_from_conf(c[idx].sum(0))["mIoU"] for c in confs_b])
        d.append(mb - ma)
    d = np.array(d)
    return dict(dif_media=float(d.mean()), ic95=(float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))),
                p_B_mayor=float((d > 0).mean()))

## 3. CLIP, regiones y embeddings

El embedding de cada región es el mismo del etiquetador: **media normalizada del recorte enmascarado y de la caja completa**, ambos cuadrados con relleno gris. Las regiones de entrenamiento son los **componentes conectados de cada clase** en el mapa de etiquetas (los `255` y el fondo quedan fuera). Se calcula una vez por dataset y se guarda en `cache_ft/`.

In [4]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(device).eval()
clip_proc  = CLIPProcessor.from_pretrained(CLIP_NAME)
clip_dtype = next(clip_model.parameters()).dtype
LOGIT_SCALE = clip_model.logit_scale.exp().item()
CLIP_DIM    = clip_model.config.projection_dim

_DUMMY_TXT = clip_proc(text=["a"], return_tensors="pt", padding=True).to(device)
_DUMMY_IMG = torch.zeros(1, 3, 224, 224, device=device, dtype=clip_dtype)
_TXT = {}

@torch.no_grad()
def embed_crops(crops):
    feats = []
    for i in range(0, len(crops), CLIP_CHUNK):
        pv = clip_proc(images=crops[i:i + CLIP_CHUNK], return_tensors="pt")["pixel_values"]
        out = clip_model(**_DUMMY_TXT, pixel_values=pv.to(device, clip_dtype))
        feats.append(F.normalize(out.image_embeds.float(), dim=-1))
    return torch.cat(feats) if feats else torch.zeros(0, CLIP_DIM, device=device)

@torch.no_grad()
def text_feat(prompt):
    if prompt not in _TXT:
        t = clip_proc(text=[prompt], return_tensors="pt", padding=True).to(device)
        out = clip_model(**t, pixel_values=_DUMMY_IMG)
        _TXT[prompt] = F.normalize(out.text_embeds.float(), dim=-1)[0]
    return _TXT[prompt]

def make_crop(img, mask, masked=True, pad=10, square=True):
    ys, xs = np.where(mask)
    y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad, img.shape[0])
    x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad, img.shape[1])
    crop = img[y0:y1, x0:x1].copy()
    if masked:
        crop[~mask[y0:y1, x0:x1]] = 127
    if square:
        h, w = crop.shape[:2]; s = max(h, w)
        canvas = np.full((s, s, 3), 127, np.uint8)
        oy, ox = (s - h) // 2, (s - w) // 2
        canvas[oy:oy + h, ox:ox + w] = crop
        crop = canvas
    return Image.fromarray(crop)

def embed_regions(img, rm):
    """(R, CLIP_DIM) float16 normalizado: media de recorte enmascarado + caja completa."""
    R = int(rm.max())
    if R == 0:
        return np.zeros((0, CLIP_DIM), np.float16)
    masks = [rm == i for i in range(1, R + 1)]
    fm = embed_crops([make_crop(img, m, masked=True) for m in masks])
    fb = embed_crops([make_crop(img, m, masked=False) for m in masks])
    return F.normalize(fm + fb, dim=-1).cpu().numpy().astype(np.float16)

def build_regions(items, out_path, force=False):
    """items: lista de (plato, ruta_etiqueta, grupo). Un registro por región (componente conectado de una clase)."""
    out_path = Path(out_path)
    if out_path.exists() and not force:
        z = np.load(out_path); tab = {k: z[k] for k in z.files}
        print("caché:", out_path, "|", len(tab["y"]), "regiones"); return tab
    embs, ys, areas, dishes, groups, t0 = [], [], [], [], [], time.time()
    for n, (d, lab_path, grp) in enumerate(items, start=1):
        rm, src = components(np.array(Image.open(lab_path)))
        if src:
            embs.append(embed_regions(load_rgb(d), rm))
            ys += src; dishes += [d] * len(src); groups += [grp] * len(src)
            areas += np.bincount(rm.ravel().astype(np.int64))[1:].tolist()
        if n % 100 == 0:
            torch.cuda.empty_cache()
            print(f"{n}/{len(items)} platos | {(time.time() - t0) / n:.2f} s/plato | {len(ys)} regiones")
    tab = dict(emb=np.concatenate(embs), y=np.array(ys, np.int16), area=np.array(areas, np.int64),
               dish=np.array(dishes), group=np.array(groups))
    np.savez_compressed(out_path, **tab)
    print("guardado:", out_path, "|", len(ys), "regiones de", len(items), "platos")
    return tab

def build_pred_regions(dish_list, map_fn):
    """Regiones PREDICHAS: componentes conectados de cada clase del mapa que devuelve map_fn(plato)."""
    out = []
    for d in dish_list:
        rm, src = components(map_fn(d))
        out.append(dict(dish=d, rm=rm, n=len(src), emb=embed_regions(load_rgb(d), rm) if src else np.zeros((0, CLIP_DIM), np.float16)))
    return out

c:\Users\mjcor\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1112 16:16:52.105000 24164 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 590/590 [00:01<00:00, 376.20it/s] 


In [5]:
suffix = f"_dbg{MAX_DISHES}" if MAX_DISHES else ""
TAB = {}
for t in DS:
    man = DS[t]["man"]
    rows = man[(man.label != "") & man.split.isin(["train", "val"])]
    if MAX_DISHES:
        rows = rows.head(MAX_DISHES)
    items = [(r.dish, DS[t]["dir"] / _p(r.label), r.split) for r in rows.itertuples()]
    TAB[t] = build_regions(items, CACHE / f"emb_{t}{suffix}.npz")

ref_rows = refs if not MAX_DISHES else refs.head(MAX_DISHES)
REF_ITEMS = [(r.dish, REFS_DIR / _p(r.label), r.group) for r in ref_rows.itertuples()]
REF_TAB = build_regions(REF_ITEMS, CACHE / f"emb_refs{suffix}.npz")

for t, tab in TAB.items():
    y, g = tab["y"], tab["group"]
    print(f"{t}: train {int((g == 'train').sum())} regiones | val {int((g == 'val').sum())} | clases con regiones en train: {len(np.unique(y[g == 'train']))} de {K}")
print("referencias:", len(REF_TAB["y"]), "regiones")

100/1242 platos | 0.19 s/plato | 220 regiones
200/1242 platos | 0.18 s/plato | 426 regiones
300/1242 platos | 0.20 s/plato | 730 regiones
400/1242 platos | 0.20 s/plato | 1018 regiones
500/1242 platos | 0.20 s/plato | 1292 regiones
600/1242 platos | 0.21 s/plato | 1611 regiones
700/1242 platos | 0.22 s/plato | 1963 regiones
800/1242 platos | 0.22 s/plato | 2280 regiones
900/1242 platos | 0.22 s/plato | 2620 regiones
1000/1242 platos | 0.23 s/plato | 3004 regiones
1100/1242 platos | 0.23 s/plato | 3359 regiones
1200/1242 platos | 0.23 s/plato | 3620 regiones
guardado: cache_ft\emb_A.npz | 3694 regiones de 1242 platos
100/1096 platos | 0.18 s/plato | 217 regiones
200/1096 platos | 0.18 s/plato | 425 regiones
300/1096 platos | 0.19 s/plato | 708 regiones
400/1096 platos | 0.20 s/plato | 989 regiones
500/1096 platos | 0.20 s/plato | 1262 regiones
600/1096 platos | 0.20 s/plato | 1570 regiones
700/1096 platos | 0.21 s/plato | 1890 regiones
800/1096 platos | 0.21 s/plato | 2199 regiones
900/

## 4. Cabeza lineal sobre CLIP

`logits = escala · emb · (T₀ + Δ)ᵀ + b`, con `T₀` = texto de CLIP de cada clase y `Δ = 0` al empezar. Sin entrenar, es **CLIP zero-shot sobre las K clases, sin metadata** (la línea base). Entrenando solo `Δ` y `b`:
- **Penalización L2-SP** (`l2sp·‖Δ‖²`): mantiene cada clase cerca de su texto. Es lo que protege a las clases con pocos ejemplos; se barre su fuerza con la validación.
- **Pesos de clase** `(mediana/n_c)^0.5`, acotados, para que las clases frecuentes no dominen.
- **Parada temprana** por pérdida de validación (sin pesos, sin suavizado).

La validación sale de las etiquetas de cada versión: sirve para elegir hiperparámetros dentro de A o de B, **no para comparar A con B**.

In [6]:
PROMPT_NAME = {"rice": "cooked rice", "leafy greens": "leafy salad greens", "wheat berry": "cooked wheat berries"}
def class_prompt(name):
    return f"a photo of {PROMPT_NAME.get(name, name)}, a type of food"

T0 = torch.stack([text_feat(class_prompt(n)) for n in CLASSES[1:]])        # (K, CLIP_DIM)

class CLIPHead(nn.Module):
    def __init__(self, T0, scale):
        super().__init__()
        self.register_buffer("T0", T0.clone()); self.scale = float(scale)
        self.delta = nn.Parameter(torch.zeros_like(T0))
        self.bias = nn.Parameter(torch.zeros(T0.shape[0], device=T0.device))
    def forward(self, x):
        return self.scale * x @ (self.T0 + self.delta).T + self.bias
    def l2sp(self):
        return (self.delta ** 2).sum(1).mean()
    def to_linear_state(self):                      # equivalente a nn.Linear(CLIP_DIM, K)
        return dict(weight=(self.scale * (self.T0 + self.delta)).detach().cpu(), bias=self.bias.detach().cpu())

def get_xy(tab, group):
    m = tab["group"] == group
    X = F.normalize(torch.tensor(tab["emb"][m].astype(np.float32)), dim=-1)
    return X, torch.tensor(tab["y"][m].astype(np.int64) - 1)          # clases 1..K -> 0..K-1

@torch.no_grad()
def head_probs(head, X):
    X = F.normalize(torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device), dim=-1)
    return head(X).softmax(-1).cpu().numpy()

def train_head(Xtr, ytr, Xva, yva, seed=0, l2sp=0.3, cw_pow=0.5, smooth=0.0, lr=1e-3, epochs=80, bs=256, patience=12):
    set_seed(seed)
    Xtr, ytr, Xva, yva = [t.to(device) for t in (Xtr, ytr, Xva, yva)]
    head = CLIPHead(T0, LOGIT_SCALE).to(device)
    cnt = torch.bincount(ytr, minlength=K).float()
    w = None
    if cw_pow:
        w = torch.where(cnt > 0, (cnt[cnt > 0].median() / cnt.clamp(min=1)) ** cw_pow, torch.ones_like(cnt)).clamp(0.25, 8.0)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.0)
    best, best_state, bad, hist = 1e9, None, 0, []
    for ep in range(epochs):
        head.train(); perm = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(head(Xtr[idx]), ytr[idx], weight=w, label_smoothing=smooth) + l2sp * head.l2sp()
            opt.zero_grad(); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            lv = head(Xva)
            vl = F.cross_entropy(lv, yva).item(); va = (lv.argmax(1) == yva).float().mean().item()
        hist.append((ep, vl, va))
        if vl < best - 1e-4:
            best, bad, best_state = vl, 0, {k: v.clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    head.load_state_dict(best_state); head.eval()
    return head, pd.DataFrame(hist, columns=["epoch", "val_loss", "val_acc"])

head0 = CLIPHead(T0, LOGIT_SCALE).to(device).eval()                      # línea base: CLIP zero-shot

In [7]:
SWEEP_L2 = [0.0, 0.3, 1.0, 3.0]
rows = []
for t in DS:
    Xtr, ytr = get_xy(TAB[t], "train"); Xva, yva = get_xy(TAB[t], "val")
    with torch.no_grad():
        acc0 = (head0(Xva.to(device)).argmax(1).cpu() == yva).float().mean().item()
    rows.append(dict(dataset=t, l2sp="zero-shot", val_loss=np.nan, val_acc=acc0, epochs=0, n_train=len(ytr), n_val=len(yva)))
    for l2 in SWEEP_L2:
        h, hist = train_head(Xtr, ytr, Xva, yva, seed=0, l2sp=l2)
        b = hist.val_loss.idxmin()
        rows.append(dict(dataset=t, l2sp=l2, val_loss=hist.val_loss[b], val_acc=hist.val_acc[b], epochs=len(hist), n_train=len(ytr), n_val=len(yva)))
sweep = pd.DataFrame(rows)
display(sweep.round(3))

num = sweep[sweep.l2sp != "zero-shot"].copy(); num["l2sp"] = num.l2sp.astype(float)
L2SP_CHOICE = {t: float(num[num.dataset == t].sort_values("val_loss").iloc[0].l2sp) for t in DS}
print("l2sp elegido por menor pérdida de validación:", L2SP_CHOICE)
# L2SP_CHOICE = {"A": 0.3, "B": 0.3}     # descomenta para fijarlo a mano

,dataset,l2sp,val_loss,val_acc,epochs,n_train,n_val
0,A,zero-shot,NaN,0.654,0,3327,367
1,A,0.0,0.549,0.856,19,3327,367
2,A,0.3,0.548,0.856,19,3327,367
3,A,1.0,0.550,0.858,19,3327,367
4,A,3.0,0.568,0.837,19,3327,367
5,B,zero-shot,NaN,0.682,0,2756,299
6,B,0.0,0.558,0.836,23,2756,299
7,B,0.3,0.556,0.839,23,2756,299
8,B,1.0,0.543,0.870,33,2756,299
9,B,3.0,0.547,0.883,33,2756,299


l2sp elegido por menor pérdida de validación: {'A': 0.3, 'B': 1.0}


In [8]:
HEADS, HISTS = {}, {}
for t in DS:
    Xtr, ytr = get_xy(TAB[t], "train"); Xva, yva = get_xy(TAB[t], "val")
    HEADS[t], HISTS[t] = [], []
    for s in SEEDS:
        h, hist = train_head(Xtr, ytr, Xva, yva, seed=s, l2sp=L2SP_CHOICE[t])
        HEADS[t].append(h); HISTS[t].append(hist)
    print(t, "| épocas por semilla:", [len(h) for h in HISTS[t]],
          "| val_acc:", [round(float(h.val_acc.iloc[h.val_loss.idxmin()]), 3) for h in HISTS[t]])

A | épocas por semilla: [19, 19, 18] | val_acc: [0.856, 0.85, 0.85]
B | épocas por semilla: [33, 23, 17] | val_acc: [0.87, 0.853, 0.853]


## 5. Evaluación en el set limpio

Dos protocolos, ambos sin metadata en la cabeza:

- **P1 · regiones de referencia.** Las regiones son los componentes de las máscaras anotadas a mano y solo se mide la clasificación: `acc_px` (precisión por píxel), `acc_region` y `recall_medio` por clase. Es el techo si la segmentación fuera perfecta.
- **P2 · regiones predichas por B1.** Se corre B1 (FoodSeg103), se toman los componentes de cada clase predicha y la cabeza les pone la clase de Nutrition5k (o "sin etiqueta" si `prob < tau`). Se mide por píxel contra la referencia: `cobertura`, `precision`, `recall_correcto`, `mIoU`.

**Diagnóstico para decidir la Parte 7:** compara `acc_px` (P1) con `precision` (P2). Si P2 cae mucho respecto a P1, el cuello de botella son las regiones de B1 y vale la pena afinar B1. Si son parecidas, el problema es la clasificación y no hace falta tocar B1.

In [9]:
from transformers import SegformerForSemanticSegmentation

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def load_b1(path):
    return SegformerForSemanticSegmentation.from_pretrained(path).to(device).eval()

@torch.no_grad()
def b1_class_map(model, dish):
    """Mapa de clases (uint8, resolución original) predicho por un SegFormer; 0 = fondo."""
    img = load_rgb(dish); H, W = img.shape[:2]
    x = torch.from_numpy(img).permute(2, 0, 1).float().div(255)[None].to(device)
    x = F.interpolate(x, size=INPUT_SIZE, mode="bilinear", align_corners=False)
    lg = model(pixel_values=(x - MEAN) / STD).logits
    lg = F.interpolate(lg, size=(H, W), mode="bilinear", align_corners=False)
    return lg.argmax(1)[0].cpu().numpy().astype(np.uint8)

REF_DISHES = list(ref_rows.dish)
GT    = {d: np.array(Image.open(REFS_DIR / _p(l))) for d, l in zip(ref_rows.dish, ref_rows.label)}
GROUP = dict(zip(ref_rows.dish, ref_rows.group))

b1 = load_b1(SEG_PATH)
P2_REGS = build_pred_regions(REF_DISHES, lambda d: b1_class_map(b1, d))
print("regiones predichas por B1:", sum(r["n"] for r in P2_REGS), "en", len(P2_REGS), "platos")

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 3318.83it/s]


regiones predichas por B1: 230 en 40 platos


In [10]:
MODELS = {"zero-shot CLIP": [head0]}
for t in DS:
    MODELS[f"cabeza {t}"] = HEADS[t]

def probs_p2(head):
    parts = [r["emb"] for r in P2_REGS if r["n"]]
    allP = head_probs(head, np.concatenate(parts)) if parts else np.zeros((0, K))
    out, k = [], 0
    for r in P2_REGS:
        out.append(allP[k:k + r["n"]]); k += r["n"]
    return out

PRB1, PRB2, CONFS = {}, {}, {}
for name, heads in MODELS.items():
    for s, h in enumerate(heads):
        PRB1[(name, s)] = head_probs(h, REF_TAB["emb"])
        PRB2[(name, s)] = probs_p2(h)
        for tau in TAUS:
            CONFS[(name, s, tau)] = np.stack([conf_for_dish(P2_REGS[i], GT[d], PRB2[(name, s)][i], tau)
                                              for i, d in enumerate(REF_DISHES)])

def evaluate_models(dish_set):
    dset = set(dish_set)
    sel1 = np.isin(REF_TAB["dish"], list(dset))
    sel2 = [i for i, d in enumerate(REF_DISHES) if d in dset]
    rows = []
    for name, heads in MODELS.items():
        for s in range(len(heads)):
            m1 = metrics_p1(PRB1[(name, s)][sel1], REF_TAB["y"][sel1], REF_TAB["area"][sel1])
            for tau in TAUS:
                m2 = metrics_from_conf(CONFS[(name, s, tau)][sel2].sum(0))
                rows.append(dict(modelo=name, semilla=s, tau=tau, **m1, **m2))
    return pd.DataFrame(rows)

def summarize(df):
    cols = ["acc_px", "acc_region", "recall_medio", "cobertura", "precision", "recall_correcto", "mIoU"]
    g = df.groupby(["modelo", "tau"])[cols]
    return g.mean().round(3).astype(str) + " ± " + g.std().fillna(0).round(3).astype(str)

GROUPS = {"todos": REF_DISHES}
for g in sorted(set(GROUP.values())):
    GROUPS[g] = [d for d in REF_DISHES if GROUP[d] == g]
EVAL = {}
for gname, dl in GROUPS.items():
    if dl:
        EVAL[gname] = evaluate_models(dl)
        print(f"\n=== {gname}: {len(dl)} platos ===")
        display(summarize(EVAL[gname]))


=== todos: 40 platos ===


acc_px     acc_region   recall_medio  \
modelo         tau                                                
cabeza A       0.0  0.879 ± 0.017  0.756 ± 0.014  0.763 ± 0.015   
               0.3  0.879 ± 0.017  0.756 ± 0.014  0.763 ± 0.015   
               0.5  0.879 ± 0.017  0.756 ± 0.014  0.763 ± 0.015   
cabeza B       0.0  0.881 ± 0.014  0.763 ± 0.018  0.766 ± 0.026   
               0.3  0.881 ± 0.014  0.763 ± 0.018  0.766 ± 0.026   
               0.5  0.881 ± 0.014  0.763 ± 0.018  0.766 ± 0.026   
zero-shot CLIP 0.0    0.666 ± 0.0    0.541 ± 0.0    0.499 ± 0.0   
               0.3    0.666 ± 0.0    0.541 ± 0.0    0.499 ± 0.0   
               0.5    0.666 ± 0.0    0.541 ± 0.0    0.499 ± 0.0   

                        cobertura      precision recall_correcto  \
modelo         tau                                                 
cabeza A       0.0    0.978 ± 0.0  0.821 ± 0.001   0.803 ± 0.001   
               0.3  0.951 ± 0.007  0.841 ± 0.005     0.8 ± 0.003   
               0.5  0.876 ± 0.019  0.885 ± 0.002   0.776 ± 0.018   
cabeza B       0.0    0.978 ± 0.0  0.831 ± 0.011    0.812 ± 0.01   
               0.3  0.956 ± 0.005  0.844 ± 0.008   0.807 ± 0.009   
               0.5  0.898 ± 0.012  0.882 ± 0.014    0.792 ± 0.01   
zero-shot CLIP 0.0    0.978 ± 0.0      0.6 ± 0.0     0.587 ± 0.0   
               0.3    0.819 ± 0.0    0.689 ± 0.0     0.564 ± 0.0   
               0.5     0.66 ± 0.0    0.787 ± 0.0     0.519 ± 0.0   

                             mIoU  
modelo         tau                 
cabeza A       0.0  0.604 ± 0.012  
               0.3  0.609 ± 0.011  
               0.5  0.598 ± 0.011  
cabeza B       0.0     0.6 ± 0.03  
               0.3  0.602 ± 0.029  
               0.5  0.596 ± 0.022  
zero-shot CLIP 0.0      0.4 ± 0.0  
               0.3    0.383 ± 0.0  
               0.5    0.359 ± 0.0


=== clean: 40 platos ===


acc_px     acc_region   recall_medio  \
modelo         tau                                                
cabeza A       0.0  0.879 ± 0.017  0.756 ± 0.014  0.763 ± 0.015   
               0.3  0.879 ± 0.017  0.756 ± 0.014  0.763 ± 0.015   
               0.5  0.879 ± 0.017  0.756 ± 0.014  0.763 ± 0.015   
cabeza B       0.0  0.881 ± 0.014  0.763 ± 0.018  0.766 ± 0.026   
               0.3  0.881 ± 0.014  0.763 ± 0.018  0.766 ± 0.026   
               0.5  0.881 ± 0.014  0.763 ± 0.018  0.766 ± 0.026   
zero-shot CLIP 0.0    0.666 ± 0.0    0.541 ± 0.0    0.499 ± 0.0   
               0.3    0.666 ± 0.0    0.541 ± 0.0    0.499 ± 0.0   
               0.5    0.666 ± 0.0    0.541 ± 0.0    0.499 ± 0.0   

                        cobertura      precision recall_correcto  \
modelo         tau                                                 
cabeza A       0.0    0.978 ± 0.0  0.821 ± 0.001   0.803 ± 0.001   
               0.3  0.951 ± 0.007  0.841 ± 0.005     0.8 ± 0.003   
               0.5  0.876 ± 0.019  0.885 ± 0.002   0.776 ± 0.018   
cabeza B       0.0    0.978 ± 0.0  0.831 ± 0.011    0.812 ± 0.01   
               0.3  0.956 ± 0.005  0.844 ± 0.008   0.807 ± 0.009   
               0.5  0.898 ± 0.012  0.882 ± 0.014    0.792 ± 0.01   
zero-shot CLIP 0.0    0.978 ± 0.0      0.6 ± 0.0     0.587 ± 0.0   
               0.3    0.819 ± 0.0    0.689 ± 0.0     0.564 ± 0.0   
               0.5     0.66 ± 0.0    0.787 ± 0.0     0.519 ± 0.0   

                             mIoU  
modelo         tau                 
cabeza A       0.0  0.604 ± 0.012  
               0.3  0.609 ± 0.011  
               0.5  0.598 ± 0.011  
cabeza B       0.0     0.6 ± 0.03  
               0.3  0.602 ± 0.029  
               0.5  0.596 ± 0.022  
zero-shot CLIP 0.0      0.4 ± 0.0  
               0.3    0.383 ± 0.0  
               0.5    0.359 ± 0.0

In [11]:
# A frente a B en el set limpio: bootstrap por plato de mIoU(B) - mIoU(A) (promedio sobre semillas)
for tau in TAUS:
    r = boot_diff([CONFS[("cabeza A", s, tau)] for s in range(len(SEEDS))],
                  [CONFS[("cabeza B", s, tau)] for s in range(len(SEEDS))])
    print(f"tau={tau}: mIoU(B) - mIoU(A) = {r['dif_media']:+.3f} | IC95 = [{r['ic95'][0]:+.3f}, {r['ic95'][1]:+.3f}] | P(B > A) = {r['p_B_mayor']:.2f}")
print("Si el IC95 contiene el 0, con estos platos no se distingue A de B: quédate con la más simple (A).")

tau=0.0: mIoU(B) - mIoU(A) = +0.003 | IC95 = [-0.015, +0.024] | P(B > A) = 0.64
tau=0.3: mIoU(B) - mIoU(A) = -0.002 | IC95 = [-0.024, +0.022] | P(B > A) = 0.41
tau=0.5: mIoU(B) - mIoU(A) = +0.005 | IC95 = [-0.011, +0.022] | P(B > A) = 0.72
Si el IC95 contiene el 0, con estos platos no se distingue A de B: quédate con la más simple (A).


In [12]:
def per_class_report(model_name, tau=TAU_MAIN, seed=0, top=30):
    conf = CONFS[(model_name, seed, tau)].sum(0)
    iou = iou_from_conf(conf)
    tp = np.diag(conf)[1:K + 1]; gt_c = conf[1:K + 1].sum(1); pr_c = conf[:, 1:K + 1].sum(0)
    rep = pd.DataFrame(dict(clase=CLASSES[1:], px_ref=gt_c, px_pred=pr_c, IoU=iou,
                            recall=np.where(gt_c > 0, tp / np.maximum(gt_c, 1), np.nan)))
    for t in DS:
        rep[f"platos_{t}"] = DS[t]["stats"].set_index("clase").platos.reindex(CLASSES[1:]).values
    return rep[rep.px_ref > 0].sort_values("px_ref", ascending=False).head(top).round(3)

BEST = "cabeza B"        # cambia según la tabla de arriba
display(per_class_report(BEST))

,clase,px_ref,px_pred,IoU,recall,platos_A,platos_B
35,leafy greens,401205,390963,0.934,0.953,226.0,186.0
53,scrambled eggs,180409,184813,0.921,0.970,76.0,73.0
14,carrot,171188,197643,0.750,0.923,87.0,74.0
12,caesar salad,109970,112266,0.856,0.932,70.0,60.0
16,cheese pizza,96964,86667,0.894,0.894,107.0,94.0
27,egg whites,95854,88133,0.860,0.887,59.0,47.0
8,berries,90399,90054,0.810,0.893,67.0,56.0
19,chicken,86644,71126,0.764,0.788,167.0,135.0
15,cauliflower,74772,71000,0.892,0.919,70.0,60.0
5,bacon,74241,71008,0.956,0.956,71.0,68.0


## 6. Guardar las cabezas

Cada `.pt` lleva la lista de clases, el nombre de CLIP, el prompt y el formato exacto del embedding, y los pesos de la capa lineal equivalente (`linear.weight`, `linear.bias`). **La lista de clases cambió** frente a la de FoodSeg103: el notebook de integración tiene que mapear estas clases a la tabla nutricional.

In [13]:
for t, hs in HEADS.items():
    for s, h in enumerate(hs):
        torch.save(dict(
            classes=CLASSES, clip_name=CLIP_NAME, tau=TAU_MAIN, seed=s, l2sp=L2SP_CHOICE[t],
            prompt="a photo of {nombre}, a type of food", logit_scale=LOGIT_SCALE,
            embedding="media normalizada de recorte enmascarado y caja completa; ambos cuadrados, relleno gris 127, pad=10",
            linear=h.to_linear_state(), state={k: v.detach().cpu() for k, v in h.state_dict().items()},
        ), f"cabeza_clip_{t}_s{s}.pt")
print("guardadas:", sorted(str(p) for p in Path(".").glob("cabeza_clip_*.pt")))

guardadas: ['cabeza_clip_A_s0.pt', 'cabeza_clip_A_s1.pt', 'cabeza_clip_A_s2.pt', 'cabeza_clip_B_s0.pt', 'cabeza_clip_B_s1.pt', 'cabeza_clip_B_s2.pt']


## 7. (Opcional) Fine-tuning de B1

Úsalo **solo si P2 cae claramente frente a P1** (las regiones de B1 son el problema). Reemplaza el clasificador de 103 clases de FoodSeg103 por uno de `K + 1` clases (fondo + Nutrition5k) y entrena con `ignore_index = 255`.

Qué hace y qué no:
- Entrena el decoder (y opcionalmente el último bloque del encoder) con `AdamW`, decaimiento polinómico y `bfloat16`; selecciona por mIoU de validación.
- Aumentos pensados para cámara cenital: volteos, rotaciones de 90°, recorte y zoom aleatorios, brillo y contraste leves.
- **No mezcla FoodSeg103.** B1 va a olvidar lo que sabía de ese dataset. Si el objetivo final incluye comida fuera de Nutrition5k, conviene mezclar después.
- Para no quedarte sin VRAM, CLIP se pasa a la CPU durante el entrenamiento.

In [14]:
RUN_B1  = False
B1_TAG  = "B"                                   # con qué dataset afinar (el que gane en la Parte 5)
B1_CFG  = dict(epochs=20, batch=8, lr=6e-5, head_lr=6e-4, unfreeze="decoder+last", cw_pow=0.5, seed=0)
# unfreeze: "decoder" | "decoder+last" (último bloque del encoder) | "all"

MEAN_CPU = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD_CPU  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class SegDS(torch.utils.data.Dataset):
    def __init__(self, rows, root_dir, train):
        self.rows = list(rows.itertuples()); self.root = Path(root_dir); self.train = train
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = load_rgb(r.dish); lab = np.array(Image.open(self.root / _p(r.label)))
        if self.train:
            if random.random() < 0.5: img, lab = img[:, ::-1], lab[:, ::-1]
            if random.random() < 0.5: img, lab = img[::-1], lab[::-1]
            k = random.randint(0, 3); img, lab = np.rot90(img, k), np.rot90(lab, k)
            H, W = lab.shape; ch, cw = int(H * random.uniform(0.6, 1.0)), int(W * random.uniform(0.6, 1.0))
            y0, x0 = random.randint(0, H - ch), random.randint(0, W - cw)
            img, lab = img[y0:y0 + ch, x0:x0 + cw], lab[y0:y0 + ch, x0:x0 + cw]
            img = np.clip(img.astype(np.float32) * random.uniform(0.8, 1.2) + random.uniform(-15, 15), 0, 255).astype(np.uint8)
        size = INPUT_SIZE[::-1]
        img = Image.fromarray(np.ascontiguousarray(img)).resize(size, Image.BILINEAR)
        lab = Image.fromarray(np.ascontiguousarray(lab)).resize(size, Image.NEAREST)
        x = (torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255) - MEAN_CPU) / STD_CPU
        return x, torch.from_numpy(np.array(lab)).long()

def train_b1(tag, epochs, batch, lr, head_lr, unfreeze, cw_pow, seed):
    set_seed(seed)
    man, ddir = DS[tag]["man"], DS[tag]["dir"]
    tr = man[(man.split == "train") & (man.label != "")]; va = man[(man.split == "val") & (man.label != "")]
    dl_tr = torch.utils.data.DataLoader(SegDS(tr, ddir, True), batch_size=batch, shuffle=True, num_workers=0, drop_last=True)
    dl_va = torch.utils.data.DataLoader(SegDS(va, ddir, False), batch_size=batch, num_workers=0)   # num_workers>0 suele fallar en Jupyter/Windows

    model = SegformerForSemanticSegmentation.from_pretrained(
        SEG_PATH, num_labels=K + 1, id2label={i: n for i, n in enumerate(CLASSES)},
        label2id={n: i for i, n in enumerate(CLASSES)}, ignore_mismatched_sizes=True).to(device)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.decode_head.parameters():
        p.requires_grad = True
    if unfreeze == "all":
        for p in model.segformer.parameters(): p.requires_grad = True
    elif unfreeze == "decoder+last":
        for m in (model.segformer.encoder.block[-1], model.segformer.encoder.layer_norm[-1]):
            for p in m.parameters(): p.requires_grad = True
    enc = [p for p in model.segformer.parameters() if p.requires_grad]
    groups = [dict(params=list(model.decode_head.parameters()), lr=head_lr)] + ([dict(params=enc, lr=lr)] if enc else [])
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    total = epochs * len(dl_tr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda it: min(1.0, (it + 1) / 100) * (1 - it / total) ** 0.9)

    px = DS[tag]["stats"].set_index("clase").pixeles.reindex(CLASSES[1:]).fillna(0).values.astype(np.float64)
    w = torch.ones(K + 1)
    if cw_pow:
        w[1:] = torch.tensor(np.where(px > 0, (np.median(px[px > 0]) / np.maximum(px, 1)) ** cw_pow, 1.0)).clamp(0.3, 5.0).float()
    w = w.to(device)

    out_dir = f"b1_ft_{tag}"; best, hist = -1.0, []
    for ep in range(epochs):
        model.train(); t0, tl = time.time(), 0.0
        for x, y in dl_tr:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                lg = model(pixel_values=x).logits
            lg = F.interpolate(lg.float(), size=y.shape[-2:], mode="bilinear", align_corners=False)
            loss = F.cross_entropy(lg, y, weight=w, ignore_index=IGNORE)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tl += loss.item()
        model.eval(); conf = np.zeros((K + 1, K + 1), np.int64)
        with torch.no_grad():
            for x, y in dl_va:
                lg = F.interpolate(model(pixel_values=x.to(device)).logits, size=y.shape[-2:], mode="bilinear", align_corners=False)
                pred, y = lg.argmax(1).cpu().numpy(), y.numpy(); v = y != IGNORE
                conf += np.bincount(y[v].astype(np.int64) * (K + 1) + pred[v], minlength=(K + 1) ** 2).reshape(K + 1, K + 1)
        tp = np.diag(conf)[1:]; gt_c, pr_c = conf[1:].sum(1), conf[:, 1:].sum(0)
        miou = float(np.nanmean(np.where(gt_c > 0, tp / np.maximum(gt_c + pr_c - tp, 1), np.nan)))
        hist.append(dict(epoca=ep, loss=tl / len(dl_tr), val_mIoU=miou))
        print(f"ep {ep + 1}/{epochs} | loss {tl / len(dl_tr):.3f} | val mIoU {miou:.3f} | {time.time() - t0:.0f} s")
        if miou > best:
            best = miou; model.save_pretrained(out_dir)
    del model, opt; torch.cuda.empty_cache()
    return out_dir, pd.DataFrame(hist)

In [15]:
if RUN_B1:
    clip_model.cpu(); torch.cuda.empty_cache()           # libera VRAM mientras entrena B1
    B1_DIR, B1_HIST = train_b1(B1_TAG, **B1_CFG)
    clip_model.to(device)
    display(B1_HIST.round(3))
    print("mejor checkpoint:", B1_DIR)

In [16]:
EVAL_B1 = False
if EVAL_B1:
    b1ft = load_b1(B1_DIR)
    head = HEADS[B1_TAG][0]
    rows = []

    # B1 base + cabeza (ya calculado en la Parte 5, semilla 0)
    for tau in TAUS:
        rows.append(dict(config="B1 base + cabeza", tau=tau, **metrics_from_conf(CONFS[(f"cabeza {B1_TAG}", 0, tau)].sum(0))))

    # B1 afinado, predicción directa (su clasificador de K+1 clases, sin cabeza)
    conf = sum(conf_from_map(b1_class_map(b1ft, d), GT[d]) for d in REF_DISHES)
    rows.append(dict(config="B1 afinado (directo)", tau=np.nan, **metrics_from_conf(conf)))

    # B1 afinado: regiones = componentes por clase predicha, reclasificadas por la cabeza
    regs = build_pred_regions(REF_DISHES, lambda d: b1_class_map(b1ft, d))
    parts = [r["emb"] for r in regs if r["n"]]
    allP = head_probs(head, np.concatenate(parts)) if parts else np.zeros((0, K))
    probs, k = [], 0
    for r in regs:
        probs.append(allP[k:k + r["n"]]); k += r["n"]
    for tau in TAUS:
        conf = sum(conf_for_dish(regs[i], GT[d], probs[i], tau) for i, d in enumerate(REF_DISHES))
        rows.append(dict(config="B1 afinado + cabeza", tau=tau, **metrics_from_conf(conf)))
    display(pd.DataFrame(rows).round(3))